In [9]:
import pandas as pd

data = pd.read_csv("./data/sc_data.csv")

/var/folders/6f/wmsnc89s2tn1gpcjgm2rbtjr0000gn/T/ipykernel_80638/3260004351.py:3: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("./data/sc_data.csv")


In [ ]:
def is_zero_to_positive_sticky(scores):
    """scores: array-like, already time-ordered, -1 rows already dropped"""
    if len(scores) < 2:
        return False
    if scores[0] != 0:
        return False
    if not (scores > 0).any():
        return False  # never transitions

    first_pos_idx = next(i for i, s in enumerate(scores) if s > 0)
    before = scores[:first_pos_idx]
    after = scores[first_pos_idx:]
    return (before == 0).all() and (after > 0).all()

# --- drop -1 values ---
df = data[data["Code-Review_score"] != -1].copy()
df = df.sort_values(["package_name", "published_at"])

# --- flag packages that qualify: 0 -> >0, then stays >0 ---
qualifies = df.groupby("package_name")["Code-Review_score"].transform(
    lambda s: is_zero_to_positive_sticky(s.values)
)

qualifying_packages = df.loc[qualifies, "package_name"].unique()

result = df.loc[
    df["package_name"].isin(qualifying_packages),
    ["github_repo", "package_name", "published_at", "Code-Review_score", "vulnerability_count"]
].copy()

result = result.sort_values(["package_name", "published_at"])

# --- QA counts ---
total_packages = data["package_name"].nunique()
after_dropping_neg1 = df["package_name"].nunique()
n_qualifying = len(qualifying_packages)

print(f"Total packages in data:                {total_packages}")
print(f"Packages with any valid (non -1) obs:  {after_dropping_neg1}")
print(f"Packages dropped entirely (all -1):    {total_packages - after_dropping_neg1}")
print(f"Packages qualifying (0 -> >0 sticky):  {n_qualifying}")
print(f"Share of valid packages qualifying:    {n_qualifying / after_dropping_neg1:.2%}")

Total packages in data:                15598
Packages with any valid (non -1) obs:  15595
Packages dropped entirely (all -1):    3
Packages qualifying (0 -> >0 sticky):  900
Share of valid packages qualifying:    5.77%


In [17]:
def decline_pre_vs_post_avg(group, min_pre_obs=1, min_post_obs=2):
    group = group.sort_values("published_at")
    scores = group["Code-Review_score"].values
    vulns = group["vulnerability_count"].values

    transition_idx = next(i for i, s in enumerate(scores) if s > 0)

    pre_vulns = vulns[:transition_idx]        # everything before the transition row
    post_vulns = vulns[transition_idx:]       # transition row onward

    if len(pre_vulns) < min_pre_obs or len(post_vulns) < min_post_obs:
        return False

    avg_before = pre_vulns.mean()
    avg_after = post_vulns.mean()

    return avg_after < avg_before

flags = result.groupby("package_name", group_keys=False).apply(
    lambda g: pd.Series(decline_pre_vs_post_avg(g), index=g.index)
)
decline_result = result.loc[flags].copy()

print(f"...with sticky 0->positive Code-Review:        {len(qualifying_packages)}")
print(f"...and avg decline (pre- vs post-transition):   {decline_result['package_name'].nunique()}")

...with sticky 0->positive Code-Review:        900
...and avg decline (pre- vs post-transition):   290


/var/folders/6f/wmsnc89s2tn1gpcjgm2rbtjr0000gn/T/ipykernel_80638/4259826914.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  flags = result.groupby("package_name", group_keys=False).apply(
